# 01 · Cohort, covariates, and outcomes

Builds the study cohort from raw MIMIC-IV v2.2 (hosp) and MIMIC-IV-ED, then the covariate, outcome, and quarterly-contact tables used by every later notebook.

**Produces (patient-level, saved to `DATA_DIR`, never committed):**
`eligibility.parquet`, `covariates.parquet`, `outcomes.parquet`, `windows.parquet`

**Produces (aggregate, saved to `results/`):** `01_cohort_counts.json`

**Paper items:** Methods, *Data source and study population*; Supplementary Methods, *Cohort definition*; Table S1 (code lists live in `src/htn_utils.py`); first rows of Figure 1A and Table S9.

| Step | What it does |
|---|---|
| 1 | Hypertension diagnoses and the surgically-correctable-cause exclusion |
| 2 | Encounters (inpatient, ED, outpatient) and outpatient blood pressure readings |
| 3 | Index date = first BP date on or after the first hypertension-coded admission |
| 4 | Eligibility: age ≥ 18, no correctable cause, no death within 90 days → **48,628** |
| 5 | Database coverage through day 730 → **at-index cohort, 37,183** |
| 6 | Hospital/ED contact around day 730 and survival status at day 730 |
| 7–9 | Covariates, cardiovascular hospitalizations, quarterly contact |
| 10 | Save |

## 0 · Setup

Edit the four paths, then run all cells. `DATA_DIR` holds patient-level files and must sit **outside** the repository.

In [ ]:
# ---- edit these ----
REPO_DIR   = "/content/drive/MyDrive/htn-bp-followup-mimic"       # this repository
MIMIC_HOSP = "/content/drive/MyDrive/MIMIC Data/mimic-iv-2.2/hosp"  # MIMIC-IV v2.2 hosp module
MIMIC_ED   = "/content/drive/MyDrive/MIMIC Data/mimic-iv-2.2/ed"    # MIMIC-IV-ED v2.2
DATA_DIR   = "/content/drive/MyDrive/htn_bp_data"                  # patient-level outputs (never commit)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install duckdb==1.3.2 comorbidipy==0.5.0

import os, sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from htn_utils import *

import numpy as np
import pandas as pd

RESULTS_DIR = os.path.join(REPO_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
con = connect(MIMIC_HOSP, MIMIC_ED)

## 1 · Hypertension diagnoses and exclusions

Hypertension and correctable secondary causes are matched by ICD-9/ICD-10 prefix on any diagnosis position (codes in Table S1 and `src/htn_utils.py`).

In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE htn_dx AS
SELECT d.subject_id, CAST(a.admittime AS DATE) AS dx_date
FROM diagnoses_icd d
JOIN admissions a ON a.subject_id = d.subject_id AND a.hadm_id = d.hadm_id
WHERE (d.icd_version = 9  AND {sql_prefix('d.icd_code', HTN_9)})
   OR (d.icd_version = 10 AND {sql_prefix('d.icd_code', HTN_10)})
''')

con.execute(f'''
CREATE OR REPLACE TABLE excl_correctable AS
SELECT DISTINCT subject_id FROM diagnoses_icd
WHERE (icd_version = 9  AND {sql_prefix('icd_code', CORRECTABLE_9)})
   OR (icd_version = 10 AND {sql_prefix('icd_code', CORRECTABLE_10)})
''')

print("patients with a hypertension diagnosis:", con.execute("SELECT COUNT(DISTINCT subject_id) FROM htn_dx").fetchone()[0])
print("patients with a correctable cause     :", con.execute("SELECT COUNT(*) FROM excl_correctable").fetchone()[0])

## 2 · Encounters and blood pressure readings

One row per patient-day of contact. A day with an admission counts as inpatient; otherwise an ED visit that did not lead to admission counts as ED; otherwise any outpatient measurement (`omr`) counts as outpatient. Blood pressure readings come from `omr`, with implausible values removed.

In [ ]:
con.execute('''
CREATE OR REPLACE TABLE encounters AS
WITH adm AS (SELECT subject_id, CAST(admittime AS DATE) AS enc_date, 1 AS rank FROM admissions),
     edv AS (SELECT subject_id, CAST(intime AS DATE) AS enc_date, 2 AS rank
             FROM ed_edstays WHERE hadm_id IS NULL),
     opt AS (SELECT DISTINCT subject_id, CAST(chartdate AS DATE) AS enc_date, 3 AS rank FROM omr),
     u   AS (SELECT * FROM adm UNION ALL SELECT * FROM edv UNION ALL SELECT * FROM opt)
SELECT subject_id, enc_date,
       CASE MIN(rank) WHEN 1 THEN 'inpatient' WHEN 2 THEN 'ed' ELSE 'outpatient' END AS enc_type
FROM u GROUP BY subject_id, enc_date
''')

con.execute('''
CREATE OR REPLACE TABLE bp AS
SELECT * FROM (
  SELECT subject_id, CAST(chartdate AS DATE) AS bp_date,
         TRY_CAST(SPLIT_PART(result_value, '/', 1) AS INTEGER) AS sbp,
         TRY_CAST(SPLIT_PART(result_value, '/', 2) AS INTEGER) AS dbp
  FROM omr WHERE result_name LIKE 'Blood Pressure%'
) WHERE sbp BETWEEN 60 AND 300 AND dbp BETWEEN 30 AND 200
''')

print(con.execute("SELECT enc_type, COUNT(*) AS n FROM encounters GROUP BY enc_type ORDER BY n DESC").df())
print("blood pressure readings:", con.execute("SELECT COUNT(*) FROM bp").fetchone()[0])

## 3 · Index date

The index date is the first blood pressure date on or after the first hypertension-coded admission. `obs_days` and `n_enc_post` describe contact after index; they are kept for reference but are **not** used to define the cohort, because they depend on information from after index.

In [ ]:
con.execute('''
CREATE OR REPLACE TABLE idx AS
WITH dx  AS (SELECT subject_id, MIN(dx_date) AS dx_date FROM htn_dx GROUP BY subject_id),
     idx AS (SELECT d.subject_id, MIN(b.bp_date) AS index_date
             FROM dx d JOIN bp b ON b.subject_id = d.subject_id AND b.bp_date >= d.dx_date
             GROUP BY d.subject_id)
SELECT i.subject_id, i.index_date,
       DATE_DIFF('day', i.index_date, MAX(e.enc_date)) AS obs_days,
       COUNT(DISTINCT e.enc_date) FILTER (WHERE e.enc_date >= i.index_date) AS n_enc_post,
       CASE WHEN p.dod IS NULL THEN NULL
            ELSE DATE_DIFF('day', i.index_date, CAST(p.dod AS DATE)) END AS days_to_death
FROM idx i
JOIN patients p ON p.subject_id = i.subject_id
LEFT JOIN encounters e ON e.subject_id = i.subject_id
GROUP BY i.subject_id, i.index_date, p.dod
''')

con.execute(f'''
CREATE OR REPLACE TABLE rx_htn AS
SELECT subject_id, COUNT(DISTINCT CAST(starttime AS DATE)) AS n_antihtn_rx
FROM prescriptions WHERE {sql_contains('drug', ANTIHTN)}
GROUP BY subject_id''')

con.execute('''
CREATE OR REPLACE TABLE bp_cnt AS
SELECT b.subject_id, COUNT(DISTINCT b.bp_date) AS n_bp_dates
FROM bp b JOIN idx i ON i.subject_id = b.subject_id
WHERE b.bp_date >= i.index_date GROUP BY b.subject_id''')

con.execute('''
CREATE OR REPLACE TABLE cand AS
SELECT i.*,
       LEAST(p.anchor_age + (EXTRACT(YEAR FROM i.index_date) - p.anchor_year), 91) AS age_index,
       (p.anchor_age + (EXTRACT(YEAR FROM i.index_date) - p.anchor_year) > 91)    AS age_censored,
       COALESCE(b.n_bp_dates, 0)   AS n_bp_dates,
       COALESCE(r.n_antihtn_rx, 0) AS n_antihtn_rx,
       (i.subject_id IN (SELECT subject_id FROM excl_correctable)) AS correctable
FROM idx i
JOIN patients p ON p.subject_id = i.subject_id
LEFT JOIN bp_cnt b ON b.subject_id = i.subject_id
LEFT JOIN rx_htn r ON r.subject_id = i.subject_id''')

cand = con.execute("SELECT * FROM cand").df()
for c in ["obs_days", "days_to_death", "n_enc_post", "n_bp_dates", "n_antihtn_rx"]:
    cand[c] = pd.to_numeric(cand[c], errors="coerce").astype(float)
cand["index_date"] = pd.to_datetime(cand["index_date"])
print(f"patients with a hypertension diagnosis and a later BP reading: {len(cand):,}")

## 4 · Eligibility

Adults (≥ 18 at index), no surgically correctable secondary cause, and no death within 90 days of index.

Two rules used in earlier versions of this analysis are **not** applied here, because both use information from after index: at least two blood pressure dates, and at least two encounters or antihypertensive prescription orders. Their flags (`bp_rule`, `enc_rule`) are kept so notebook 02 can confirm that no patient affected by them reaches the landmark population (Supplementary Methods).

In [ ]:
flow = [("Hypertension diagnosis with a later BP reading", len(cand))]
e = cand[cand.age_index >= 18];                                    flow.append(("Age 18 or older at index", len(e)))
e = e[~e.correctable.astype(bool)];                                flow.append(("No surgically correctable cause", len(e)))
e = e[e.days_to_death.isna() | (e.days_to_death >= DEATH_EXCL_DAYS)]; flow.append(("No death within 90 days of index", len(e)))
e = e.copy()

for step, n in flow:
    print(f"{n:>8,}  {step}")
print()
check("Eligible at index", len(e), 48628)

# rules from earlier versions, kept as flags only (not applied)
e["bp_rule"]  = e.n_bp_dates >= 2
e["enc_rule"] = (e.n_enc_post >= 2) | (e.n_antihtn_rx >= 2)

## 5 · Database coverage (at-index cohort)

MIMIC-IV shifts each patient's dates and reports only a three-year `anchor_year_group` for the real calendar period. Coverage ends at the latest date the hospital data can be guaranteed to include: December 31 of the shifted year matching real year 2020, assuming the **latest** year in the patient's group (conservative bound). A patient enters the at-index cohort when coverage reaches day 730, regardless of whether they returned.

In [ ]:
pt = con.execute("SELECT subject_id, anchor_year, anchor_year_group FROM patients").df()
g = pt.anchor_year_group.str.extract(r"(\d{4})\s*-\s*(\d{4})").astype(int)
pt["g_lo"], pt["g_hi"] = g[0], g[1]

elig = e.merge(pt[["subject_id", "anchor_year", "g_lo", "g_hi"]], on="subject_id", how="left")
elig["admin_end"] = pd.to_datetime((elig.anchor_year + HOSP_DATA_END_YEAR - elig.g_hi).astype(str) + "-12-31")
elig["admin_day"] = (elig.admin_end - elig.index_date).dt.days       # last covered day, counted from index
elig["at_index_cohort"] = elig.admin_day >= LANDMARK_DAY

check("At-index cohort (coverage through day 730)", int(elig.at_index_cohort.sum()), 37183)
print(f"         without coverage through day 730: {int((~elig.at_index_cohort).sum()):,}")

## 6 · Contact around day 730 and survival status

For each patient, from encounters between index and the end of coverage:

- `last_hosp_day`: last inpatient or ED contact (death capture runs 365 days past it)
- `hosp_366_730`: any inpatient or ED contact on days 366–730
- `any_after_730` / `last_any_day`: any record after day 730

Survival at day 730 is then classified in this order: died by day 730; alive, confirmed by a record after day 730; alive, confirmed by hospital/ED contact on days 366–730; alive, confirmed by a death recorded after day 730; otherwise unknown. Notebook 02 uses these to build the primary landmark population and the day-730-information sample.

In [ ]:
ids = elig[["subject_id", "index_date", "admin_day"]].copy()
ids["index_date"] = ids.index_date.dt.date
con.register("ids", ids)

q = con.execute('''
  WITH e AS (
    SELECT i.subject_id, e.enc_type, i.admin_day,
           DATE_DIFF('day', CAST(i.index_date AS DATE), CAST(e.enc_date AS DATE)) AS dd
    FROM ids i JOIN encounters e ON e.subject_id = i.subject_id)
  SELECT subject_id,
         MAX(CASE WHEN enc_type IN ('inpatient','ed') AND dd BETWEEN 0 AND admin_day THEN dd END) AS last_hosp_day,
         MAX(CASE WHEN enc_type IN ('inpatient','ed') AND dd BETWEEN 366 AND 730 THEN 1 ELSE 0 END) AS hosp_366_730,
         MAX(CASE WHEN dd > 730 AND dd <= admin_day THEN 1 ELSE 0 END)                            AS any_after_730,
         MAX(CASE WHEN dd BETWEEN 0 AND admin_day THEN dd END)                                     AS last_any_day
  FROM e GROUP BY subject_id''').df()

elig = elig.merge(q, on="subject_id", how="left")
for c in ["hosp_366_730", "any_after_730"]:
    elig[c] = elig[c].fillna(0).astype(int)
elig["last_any_day"] = pd.to_numeric(elig.last_any_day, errors="coerce").astype(float)
elig["last_hosp_day"] = pd.to_numeric(elig.last_hosp_day, errors="coerce").astype(float)
elig["capture_end_day"] = elig.last_hosp_day + DEATH_CAPTURE_DAYS    # deaths captured up to here

def status730(d):
    died        = d.days_to_death.le(LANDMARK_DAY).fillna(False)
    after       = d.any_after_730.gt(0) | d.last_any_day.gt(LANDMARK_DAY).fillna(False)
    hosp        = d.hosp_366_730.gt(0)
    death_after = d.days_to_death.gt(LANDMARK_DAY).fillna(False)
    return np.select([died, after, hosp, death_after],
                     ["died by day 730",
                      "alive: record after day 730",
                      "alive: hospital/ED contact days 366-730",
                      "alive: death recorded after day 730"],
                     "survival unknown")

elig["status730"] = status730(elig)
print("Survival status at day 730, at-index cohort:")
print(elig.loc[elig.at_index_cohort, "status730"].value_counts().to_string())

**Death capture check (Supplementary Methods).** Deaths in MIMIC-IV are captured up to about one year after the last hospital discharge. This prints how long after the last hospital or ED contact recorded deaths occurred.

*Note: the paper's "99% within 355 days" was computed in an earlier cohort definition; this cell recomputes it in the eligible cohort, so the figure may differ slightly.*

In [ ]:
lastc = con.execute('''
  SELECT subject_id, MAX(t) AS last_hosp FROM (
    SELECT subject_id, CAST(dischtime AS TIMESTAMP) AS t FROM admissions
    UNION ALL SELECT subject_id, CAST(outtime AS TIMESTAMP) FROM ed_edstays)
  GROUP BY subject_id''').df()
dod = con.execute("SELECT subject_id, dod FROM patients WHERE dod IS NOT NULL").df()
d = elig[["subject_id"]].merge(dod, on="subject_id").merge(lastc, on="subject_id", how="left")
d["lag"] = (pd.to_datetime(d.dod) - pd.to_datetime(d.last_hosp).dt.normalize()).dt.days
print(f"recorded deaths: {len(d):,}")
print(f"99th percentile, days from last hospital/ED contact to death: {d.lag.quantile(0.99):.0f}")

## 7 · Covariates

Built for all eligible patients (each patient's values depend only on their own index date, so later notebooks take subsets).

- Demographics: most common race, insurance, language, and marital status across admissions; race grouped as White, Black, Hispanic, Asian, Other/Unknown
- Blood pressure, creatinine, glycated hemoglobin (`labevents` items 50912, 50852), body mass index, and antihypertensive drug count: **365 days before to 90 days after index.** This window is the Table S3 sensitivity analysis (*Covariates averaged from −365 to +90 days*); the primary analysis rebuilds these with pre-index windows in notebook 06.
- eGFR: 2021 CKD-EPI creatinine equation without race
- Prior emergency, inpatient, and total encounters: before index
- Elixhauser comorbidities (Quan coding, van Walraven score): diagnoses from admissions on or before index

In [ ]:
cohort_ids = elig[["subject_id"]].copy()
con.register("cohort_ids", cohort_ids)
con.execute("CREATE OR REPLACE TABLE cohort AS SELECT c.* FROM cand c JOIN cohort_ids USING (subject_id)")

con.execute('''
CREATE OR REPLACE TABLE cov_demo AS
SELECT c.subject_id, p.gender,
       MODE(a.race) AS race_raw, MODE(a.insurance) AS insurance,
       MODE(a.language) AS language, MODE(a.marital_status) AS marital_status
FROM cohort c
JOIN patients p ON p.subject_id = c.subject_id
LEFT JOIN admissions a ON a.subject_id = c.subject_id
GROUP BY c.subject_id, p.gender''')

con.execute('''
CREATE OR REPLACE TABLE cov_bp AS
SELECT c.subject_id, AVG(b.sbp) AS sbp_baseline, AVG(b.dbp) AS dbp_baseline
FROM cohort c JOIN bp b ON b.subject_id = c.subject_id
WHERE b.bp_date BETWEEN c.index_date - 365 AND c.index_date + 90
GROUP BY c.subject_id''')

con.execute('''
CREATE OR REPLACE TABLE cov_util AS
SELECT c.subject_id,
       COUNT(*) FILTER (WHERE e.enc_type = 'ed'        AND e.enc_date < c.index_date) AS n_ed_prior,
       COUNT(*) FILTER (WHERE e.enc_type = 'inpatient' AND e.enc_date < c.index_date) AS n_inp_prior,
       COUNT(*) FILTER (WHERE e.enc_date < c.index_date)                              AS n_enc_prior
FROM cohort c LEFT JOIN encounters e ON e.subject_id = c.subject_id
GROUP BY c.subject_id''')

con.execute(f'''
CREATE OR REPLACE TABLE cov_rx AS
SELECT c.subject_id, COUNT(DISTINCT LOWER(pr.drug)) AS n_antihtn_drugs
FROM cohort c JOIN prescriptions pr ON pr.subject_id = c.subject_id
WHERE {sql_contains('pr.drug', ANTIHTN)}
  AND CAST(pr.starttime AS DATE) BETWEEN c.index_date - 365 AND c.index_date + 90
GROUP BY c.subject_id''')
print("demographics, blood pressure, prior use, prescriptions done")

In [ ]:
# labs, BMI, eGFR (slowest cell: labevents is the largest table)
con.execute('''
CREATE OR REPLACE TABLE cov_lab AS
WITH l AS (
  SELECT subject_id, itemid, valuenum, CAST(charttime AS DATE) AS lab_date
  FROM labevents WHERE itemid IN (50912, 50852) AND valuenum IS NOT NULL)
SELECT c.subject_id,
       AVG(l.valuenum) FILTER (WHERE l.itemid = 50912) AS creatinine,
       AVG(l.valuenum) FILTER (WHERE l.itemid = 50852) AS hba1c
FROM cohort c JOIN l ON l.subject_id = c.subject_id
WHERE l.lab_date BETWEEN c.index_date - 365 AND c.index_date + 90
GROUP BY c.subject_id''')

con.execute('''
CREATE OR REPLACE TABLE cov_bmi AS
SELECT c.subject_id, AVG(TRY_CAST(o.result_value AS DOUBLE)) AS bmi
FROM cohort c JOIN omr o ON o.subject_id = c.subject_id
WHERE o.result_name LIKE 'BMI%'
  AND CAST(o.chartdate AS DATE) BETWEEN c.index_date - 365 AND c.index_date + 90
  AND TRY_CAST(o.result_value AS DOUBLE) BETWEEN 12 AND 80
GROUP BY c.subject_id''')

# 2021 CKD-EPI creatinine equation, without race
con.execute('''
CREATE OR REPLACE TABLE cov_egfr AS
SELECT c.subject_id, l.creatinine,
       CASE WHEN l.creatinine IS NULL THEN NULL ELSE
         142
         * POWER(LEAST(l.creatinine / CASE WHEN d.gender = 'F' THEN 0.7 ELSE 0.9 END, 1),
                 CASE WHEN d.gender = 'F' THEN -0.241 ELSE -0.302 END)
         * POWER(GREATEST(l.creatinine / CASE WHEN d.gender = 'F' THEN 0.7 ELSE 0.9 END, 1), -1.200)
         * POWER(0.9938, c.age_index)
         * CASE WHEN d.gender = 'F' THEN 1.012 ELSE 1.0 END
       END AS egfr
FROM cohort c
JOIN cov_demo d USING (subject_id)
LEFT JOIN cov_lab l USING (subject_id)''')
print("labs, BMI, eGFR done")

In [ ]:
# Elixhauser comorbidities (comorbidipy 0.5.0, Quan coding, van Walraven weights)
import pandas.core.common as pdc
from pandas.errors import SettingWithCopyWarning
pdc.SettingWithCopyWarning = SettingWithCopyWarning   # comorbidipy 0.5 imports it from the pre-pandas-2.0 location
from comorbidipy import comorbidity

dx = con.execute('''
SELECT c.subject_id AS id, d.icd_code AS code, d.icd_version
FROM cohort c
JOIN admissions a ON a.subject_id = c.subject_id
JOIN diagnoses_icd d ON d.subject_id = a.subject_id AND d.hadm_id = a.hadm_id
WHERE CAST(a.admittime AS DATE) <= c.index_date''').df()

parts = []
for ver, tag in [(9, "icd9"), (10, "icd10")]:
    sub = dx[dx.icd_version == ver][["id", "code"]].drop_duplicates().copy()
    if len(sub):
        sub["age"] = 60   # required by the API, unused by van Walraven weighting
        parts.append(comorbidity(sub, id="id", code="code", age=None, score="elixhauser",
                                 icd=tag, variant="quan", weighting="vw", assign0=False))

# patients spanning the 2015 ICD-9 -> ICD-10 transition: a category counts if flagged in either era
elix = pd.concat(parts).groupby("id").max().reset_index()
print("Elixhauser table:", elix.shape)

In [ ]:
con.register("elix_df", elix)
con.execute('''
CREATE OR REPLACE TABLE covariates AS
SELECT c.subject_id, c.index_date, c.age_index, c.age_censored,
       d.gender, d.insurance, d.language, d.marital_status,
       CASE WHEN d.race_raw LIKE 'WHITE%'    THEN 'White'
            WHEN d.race_raw LIKE 'BLACK%'    THEN 'Black'
            WHEN d.race_raw LIKE 'HISPANIC%' THEN 'Hispanic'
            WHEN d.race_raw LIKE 'ASIAN%'    THEN 'Asian'
            ELSE 'Other/Unknown' END AS race,
       v.sbp_baseline, v.dbp_baseline,
       l.creatinine, g.egfr, b.bmi, l.hba1c,
       (l.hba1c IS NULL) AS hba1c_missing,
       (b.bmi   IS NULL) AS bmi_missing,
       COALESCE(u.n_ed_prior, 0)      AS n_ed_prior,
       COALESCE(u.n_inp_prior, 0)     AS n_inp_prior,
       COALESCE(u.n_enc_prior, 0)     AS n_enc_prior,
       COALESCE(r.n_antihtn_drugs, 0) AS n_antihtn_drugs,
       e.comorbidity_score AS elixhauser_vw,
       e.* EXCLUDE (id, comorbidity_score)
FROM cohort c
LEFT JOIN cov_demo d USING (subject_id)
LEFT JOIN cov_bp   v USING (subject_id)
LEFT JOIN cov_egfr g USING (subject_id)
LEFT JOIN cov_bmi  b USING (subject_id)
LEFT JOIN cov_lab  l USING (subject_id)
LEFT JOIN cov_util u USING (subject_id)
LEFT JOIN cov_rx   r USING (subject_id)
LEFT JOIN elix_df  e ON e.id = c.subject_id''')

covariates = con.execute("SELECT * FROM covariates").df()
print(f"covariates: {covariates.shape}")
print("missing, at-index cohort (%):")
ai = covariates.subject_id.isin(elig.loc[elig.at_index_cohort, "subject_id"])
print((covariates.loc[ai, ["egfr", "hba1c", "bmi"]].isna().mean() * 100).round(1).to_string())

## 8 · Cardiovascular hospitalizations

A cardiovascular hospitalization is an admission with myocardial infarction, stroke, or heart failure as the **principal** diagnosis (`seq_num = 1`; codes in Table S1). `prior_mace` flags one on or before index. The landmark analysis (notebook 02) recounts events from day 730, and the any-position definition used in Table S8 is built in notebook 06.

*Column names keep the `mace` prefix used throughout the original code.*

In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE mace_events AS
SELECT c.subject_id, CAST(a.admittime AS DATE) AS event_date,
       MAX(CASE WHEN (d.icd_version=9  AND {sql_prefix('d.icd_code', MI_9)})
                  OR (d.icd_version=10 AND {sql_prefix('d.icd_code', MI_10)})     THEN 1 ELSE 0 END) AS is_mi,
       MAX(CASE WHEN (d.icd_version=9  AND {sql_prefix('d.icd_code', STROKE_9)})
                  OR (d.icd_version=10 AND {sql_prefix('d.icd_code', STROKE_10)}) THEN 1 ELSE 0 END) AS is_stroke,
       MAX(CASE WHEN (d.icd_version=9  AND {sql_prefix('d.icd_code', HF_9)})
                  OR (d.icd_version=10 AND {sql_prefix('d.icd_code', HF_10)})     THEN 1 ELSE 0 END) AS is_hf
FROM cohort c
JOIN admissions a ON a.subject_id = c.subject_id
JOIN diagnoses_icd d ON d.subject_id = a.subject_id AND d.hadm_id = a.hadm_id
WHERE d.seq_num = 1
  AND ((d.icd_version=9  AND {sql_prefix('d.icd_code', CV_9)})
    OR (d.icd_version=10 AND {sql_prefix('d.icd_code', CV_10)}))
GROUP BY c.subject_id, CAST(a.admittime AS DATE)''')

con.execute('''
CREATE OR REPLACE TABLE outcomes AS
WITH prior AS (
  SELECT c.subject_id, MAX(CASE WHEN m.event_date <= c.index_date THEN 1 ELSE 0 END) AS prior_mace
  FROM cohort c LEFT JOIN mace_events m USING (subject_id) GROUP BY c.subject_id),
first_ev AS (
  SELECT c.subject_id, MIN(m.event_date) AS mace_date
  FROM cohort c JOIN mace_events m USING (subject_id)
  WHERE m.event_date > c.index_date GROUP BY c.subject_id),
lastenc AS (
  SELECT c.subject_id, MAX(e.enc_date) AS last_enc
  FROM cohort c JOIN encounters e USING (subject_id)
  WHERE e.enc_date >= c.index_date GROUP BY c.subject_id)
SELECT c.subject_id, c.index_date, p.prior_mace, f.mace_date,
       CASE WHEN pt.dod IS NULL THEN NULL ELSE CAST(pt.dod AS DATE) END AS death_date,
       l.last_enc,
       CASE WHEN f.mace_date IS NOT NULL THEN 1
            WHEN pt.dod IS NOT NULL THEN 2 ELSE 0 END AS event_type,
       DATE_DIFF('day', c.index_date,
         COALESCE(f.mace_date, CAST(pt.dod AS DATE), l.last_enc)) AS time_days
FROM cohort c
LEFT JOIN prior p USING (subject_id)
LEFT JOIN first_ev f USING (subject_id)
LEFT JOIN lastenc l USING (subject_id)
JOIN patients pt ON pt.subject_id = c.subject_id''')

outcomes = con.execute("SELECT * FROM outcomes").df()
mace_events = con.execute("SELECT * FROM mace_events").df()
print(f"outcomes: {outcomes.shape} | cardiovascular admissions (all dates): {len(mace_events):,}")

## 9 · Quarterly contact counts

Twenty 91-day quarters from index, each with the number of distinct days with an encounter (`n_visits`), a blood pressure reading (`n_bp`), and an antihypertensive prescription order (`n_rx`), ending at death.

These **summed-by-type** counts are used only for the *alternative class definition* sensitivity analysis (Table S3). The primary trajectory input counts each contact day once and is built in notebook 02.

In [ ]:
# slow cell: 20 quarters x all eligible patients, three range joins
con.execute(f'''
CREATE OR REPLACE TABLE windows AS
WITH grid AS (
  SELECT c.subject_id, CAST(q.q AS INTEGER) AS q,
         c.index_date + CAST(q.q * 91 AS INTEGER)           AS q_start,
         c.index_date + CAST((q.q + 1) * 91 - 1 AS INTEGER) AS q_end,
         o.death_date
  FROM cohort c
  JOIN outcomes o USING (subject_id)
  CROSS JOIN (SELECT UNNEST(RANGE(0, 20)) AS q) q)
SELECT g.subject_id, g.q, g.q_start, g.q_end,
       COUNT(DISTINCT e.enc_date)                   AS n_visits,
       COUNT(DISTINCT b.bp_date)                    AS n_bp,
       COUNT(DISTINCT CAST(pr.starttime AS DATE))   AS n_rx
FROM grid g
LEFT JOIN encounters e ON e.subject_id = g.subject_id AND e.enc_date BETWEEN g.q_start AND g.q_end
LEFT JOIN bp b         ON b.subject_id = g.subject_id AND b.bp_date  BETWEEN g.q_start AND g.q_end
LEFT JOIN prescriptions pr ON pr.subject_id = g.subject_id
     AND CAST(pr.starttime AS DATE) BETWEEN g.q_start AND g.q_end
     AND {sql_contains('pr.drug', ANTIHTN)}
WHERE g.death_date IS NULL OR g.q_start <= g.death_date
GROUP BY g.subject_id, g.q, g.q_start, g.q_end''')

con.execute('''
CREATE OR REPLACE TABLE windows AS
SELECT *, TRUE AS in_followup, n_visits / 1.0 AS visit_adherence, (n_visits = 0) AS gap_quarter
FROM windows''')

windows = con.execute("SELECT * FROM windows").df()
print(f"windows: {windows.shape}")

## 10 · Save

Patient-level tables go to `DATA_DIR`. Only counts go to `results/`.

In [ ]:
elig_cols = ["subject_id", "index_date", "age_index", "anchor_year", "g_lo", "g_hi",
             "admin_end", "admin_day", "at_index_cohort",
             "obs_days", "days_to_death", "n_bp_dates", "n_enc_post", "n_antihtn_rx", "bp_rule", "enc_rule",
             "last_hosp_day", "last_any_day", "hosp_366_730", "any_after_730", "capture_end_day", "status730"]

save_df(elig[elig_cols], DATA_DIR, "eligibility")
save_df(covariates,      DATA_DIR, "covariates")
save_df(outcomes,        DATA_DIR, "outcomes")
save_df(mace_events,     DATA_DIR, "mace_events")
save_df(windows,         DATA_DIR, "windows")

save_json({
    "flow": [{"step": s, "n": int(n)} for s, n in flow],
    "eligible_at_index": int(len(elig)),
    "at_index_cohort": int(elig.at_index_cohort.sum()),
    "status730_at_index_cohort": elig.loc[elig.at_index_cohort, "status730"].value_counts().to_dict(),
    "rules": {
        "index_date": "first BP date on or after the first hypertension-coded admission",
        "coverage": f"Dec 31 of (anchor_year + {HOSP_DATA_END_YEAR} - upper year of anchor_year_group) >= index + 730 days",
        "death_capture_days_after_last_hospital_or_ED_contact": DEATH_CAPTURE_DAYS,
        "followup_end": "earliest of: end of coverage, day 1460, last hospital/ED contact + 365, death",
        "rules_not_applied": ">=2 BP dates; >=2 encounters or antihypertensive orders (both use post-index information)",
    },
}, RESULTS_DIR, "01_cohort_counts")